In [1]:
%pip install neo4j SPARQLWrapper pandas dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from neo4j import GraphDatabase
from SPARQLWrapper import SPARQLWrapper, JSON
import time
import os
from dotenv import load_dotenv

# --- KONFIGURASI ---
# Load environment variables from .env file
load_dotenv()

URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
USER = os.getenv("NEO4J_USER", "neo4j")
PASSWORD = os.getenv("NEO4J_PASSWORD", "your_password") # Ganti Password
WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"

def get_driver():
    return GraphDatabase.driver(URI, auth=(USER, PASSWORD))

# 1. Fungsi untuk mengambil ID dari Neo4j
def get_anime_ids_from_neo4j(driver):
    query = "MATCH (a:Anime) RETURN a.malAnimeId AS id"
    with driver.session() as session:
        result = session.run(query)
        return [record["id"] for record in result]

# 2. Fungsi Query SPARQL (Batching ID)
def fetch_wikidata_batch(mal_ids):
    # Mengubah list ID menjadi string untuk query SPARQL: "1" "5114" ...
    values_str = " ".join([f'"{id}"' for id in mal_ids])
    
    sparql_query = f"""
    SELECT ?malId ?website ?imdbId
    WHERE {{
        VALUES ?malId {{ {values_str} }}
        ?anime wdt:P4086 ?malId .
        OPTIONAL {{ ?anime wdt:P856 ?website . }}
        OPTIONAL {{ ?anime wdt:P345 ?imdbId . }}
    }}
    """
    
    sparql = SPARQLWrapper(WIKIDATA_ENDPOINT)
    sparql.setQuery(sparql_query)
    sparql.setReturnFormat(JSON)
    
    try:
        results = sparql.query().convert()
        bindings = results["results"]["bindings"]
        
        # Bersihkan hasil menjadi list of dictionaries
        enriched_data = []
        for b in bindings:
            item = {
                "malId": int(b["malId"]["value"]),                
                "website": b.get("website", {}).get("value", None), 
                "imdbUrl": f"https://www.imdb.com/title/{b['imdbId']['value']}" if "imdbId" in b else None
            }
            enriched_data.append(item)
        return enriched_data
        
    except Exception as e:
        print(f"SPARQL Error: {e}")
        return []

# 3. Fungsi Update ke Neo4j (UNWIND)
def update_neo4j_batch(driver, data_batch):
    if not data_batch: return
    
    cypher_query = """
    UNWIND $batch AS row
    MATCH (a:Anime {malAnimeId: row.malId})
    SET a.imdbUrl = row.imdbUrl,
        a.officialWebsite = row.website
    """
    with driver.session() as session:
        session.run(cypher_query, batch=data_batch)

# --- EKSEKUSI UTAMA ---
def main():
    driver = get_driver()
    
    print("1. Mengambil ID dari Neo4j...")
    all_ids = get_anime_ids_from_neo4j(driver)
    total = len(all_ids)
    print(f"   Ditemukan {total} anime.")
    
    # Proses dalam chunk (misal per 200 ID agar URL SPARQL tidak kepanjangan)
    CHUNK_SIZE = 200
    
    print("2. Memulai proses Enrichment...")
    for i in range(0, total, CHUNK_SIZE):
        chunk_ids = all_ids[i : i + CHUNK_SIZE]
        
        # A. Tanya Wikidata
        enriched_data = fetch_wikidata_batch(chunk_ids)
        
        # B. Update Neo4j
        if enriched_data:
            # print(enriched_data)
            update_neo4j_batch(driver, enriched_data)
            
        print(f"   Progress: {min(i + CHUNK_SIZE, total)}/{total} anime diproses...")
        
        # Penting: Beri jeda agar tidak diblokir Wikidata
        time.sleep(1) 

    print("\n✅ Integrasi Selesai!")
    driver.close()

if __name__ == "__main__":
    main()

1. Mengambil ID dari Neo4j...
   Ditemukan 19931 anime.
2. Memulai proses Enrichment...
   Ditemukan 19931 anime.
2. Memulai proses Enrichment...
   Progress: 200/19931 anime diproses...
   Progress: 200/19931 anime diproses...
   Progress: 400/19931 anime diproses...
   Progress: 400/19931 anime diproses...
   Progress: 600/19931 anime diproses...
   Progress: 600/19931 anime diproses...
   Progress: 800/19931 anime diproses...
   Progress: 800/19931 anime diproses...
   Progress: 1000/19931 anime diproses...
   Progress: 1000/19931 anime diproses...
   Progress: 1200/19931 anime diproses...
   Progress: 1200/19931 anime diproses...
   Progress: 1400/19931 anime diproses...
   Progress: 1400/19931 anime diproses...
   Progress: 1600/19931 anime diproses...
   Progress: 1600/19931 anime diproses...
   Progress: 1800/19931 anime diproses...
   Progress: 1800/19931 anime diproses...
   Progress: 2000/19931 anime diproses...
   Progress: 2000/19931 anime diproses...
   Progress: 2200/1993